# 04 — Analysis & Visuals

**Purpose:** turn the per-question scores from `03_scoring.ipynb` into per-category scorecards for
each model, then visualise them — first one model at a time, then as a single side-by-side
comparison across all evaluated models.


## 1. Scorecard — original pilot run

`generate_scorecard.py` is the earliest version of this step, built against `data/scores.csv` from
the project's original 15-question pilot test set (before it was expanded to 42 questions and moved
to OpenRouter). Kept here for reference alongside the per-model scorecards below.


In [ ]:
import pandas as pd

# Load the completed scores
scores_df = pd.read_csv("data/scores.csv")

# Makes sure the score column is treated as numbers, not text
scores_df["score"] = pd.to_numeric(scores_df["score"], errors="coerce")

# Overall summary
overall_average = scores_df["score"].mean()
total_tests = len(scores_df)
pass_count = (scores_df["score"] == 2).sum()
partial_count = (scores_df["score"] == 1).sum()
fail_count = (scores_df["score"] == 0).sum()

print("=== OVERALL SCORECARD ===")
print(f"Total tests: {total_tests}")
print(f"Average score: {overall_average:.2f} / 2")
print(f"Fully passed (2): {pass_count} ({pass_count/total_tests*100:.0f}%)")
print(f"Partial (1): {partial_count} ({partial_count/total_tests*100:.0f}%)")
print(f"Failed (0): {fail_count} ({fail_count/total_tests*100:.0f}%)")

# Breakdown by category
print("\n=== BREAKDOWN BY CATEGORY ===")
category_summary = scores_df.groupby("category")["score"].agg(["mean", "count"])
category_summary = category_summary.rename(columns={"mean": "average_score", "count": "num_tests"})
print(category_summary)

# Save the category summary as its own CSV too
category_summary.to_csv("data/scorecard_summary.csv")

print("\nDone! Category summary also saved to data/scorecard_summary.csv")


## 2. Scorecard — Gemma (automated LLM-judge scores)


In [ ]:
import pandas as pd

scores_df = pd.read_csv("data/scores_gemma_auto.csv", encoding="utf-8", encoding_errors="replace")
scores_df["score"] = pd.to_numeric(scores_df["score"], errors="coerce")

overall_average = scores_df["score"].mean()
total_tests = scores_df["score"].notna().sum()
excluded_count = scores_df["score"].isna().sum()
pass_count = (scores_df["score"] == 2).sum()
partial_count = (scores_df["score"] == 1).sum()
fail_count = (scores_df["score"] == 0).sum()

print("=== GEMMA OVERALL SCORECARD (Automated, LLM-Judged) ===")
print(f"Total tests scored: {total_tests} (excluded/N/A: {excluded_count})")
print(f"Average score: {overall_average:.2f} / 2")
print(f"Fully passed (2): {pass_count} ({pass_count/total_tests*100:.0f}%)")
print(f"Partial (1): {partial_count} ({partial_count/total_tests*100:.0f}%)")
print(f"Failed (0): {fail_count} ({fail_count/total_tests*100:.0f}%)")

category_summary = scores_df.groupby("category")["score"].agg(["mean", "count"])
category_summary = category_summary.rename(columns={"mean": "average_score", "count": "num_tests"})
print("\n=== BREAKDOWN BY CATEGORY ===")
print(category_summary)

category_summary.to_csv("data/scorecard_summary_gemma.csv")
print("\nDone! Saved to data/scorecard_summary_gemma.csv (now reflects automated scoring)")


## 3. Scorecard — Nemotron (manual scores)


In [ ]:
import pandas as pd

scores_df = pd.read_csv("data/scores_nemotron.csv", encoding="utf-8", encoding_errors="replace")
scores_df["score"] = pd.to_numeric(scores_df["score"], errors="coerce")

overall_average = scores_df["score"].mean()
total_tests = scores_df["score"].notna().sum()
excluded_count = scores_df["score"].isna().sum()
pass_count = (scores_df["score"] == 2).sum()
partial_count = (scores_df["score"] == 1).sum()
fail_count = (scores_df["score"] == 0).sum()

print("=== NEMOTRON OVERALL SCORECARD ===")
print(f"Total tests scored: {total_tests} (excluded/N/A: {excluded_count})")
print(f"Average score: {overall_average:.2f} / 2")
print(f"Fully passed (2): {pass_count} ({pass_count/total_tests*100:.0f}%)")
print(f"Partial (1): {partial_count} ({partial_count/total_tests*100:.0f}%)")
print(f"Failed (0): {fail_count} ({fail_count/total_tests*100:.0f}%)")

category_summary = scores_df.groupby("category")["score"].agg(["mean", "count"])
category_summary = category_summary.rename(columns={"mean": "average_score", "count": "num_tests"})
print("\n=== BREAKDOWN BY CATEGORY ===")
print(category_summary)

category_summary.to_csv("data/scorecard_summary_nemotron.csv")
print("\nDone! Saved to data/scorecard_summary_nemotron.csv")


## 4. Scorecard — OpenRouter (general/pre-split run)

This scorecard predates splitting results out per model (Gemma / Nemotron / gpt-oss) and reflects
an earlier combined OpenRouter run. It's kept for historical reference; the final side-by-side
comparison in section 7 below uses the per-model scorecards instead.


In [ ]:
import pandas as pd

# Load your completed OpenRouter scores
scores_df = pd.read_csv("data/scores_openrouter.csv", encoding="utf-8", encoding_errors="replace")

# Make sure the score column is treated as numbers, not text
scores_df["score"] = pd.to_numeric(scores_df["score"], errors="coerce")

# --- Overall summary ---
# Note: N/A rows (corrupted responses) are automatically excluded since
# pd.to_numeric turns "N/A" into a missing value, and .mean()/.sum() skip those
overall_average = scores_df["score"].mean()
total_tests = scores_df["score"].notna().sum()  # count only valid, scored rows
excluded_count = scores_df["score"].isna().sum()
pass_count = (scores_df["score"] == 2).sum()
partial_count = (scores_df["score"] == 1).sum()
fail_count = (scores_df["score"] == 0).sum()

print("=== OPENROUTER OVERALL SCORECARD ===")
print(f"Total tests scored: {total_tests} (excluded due to corruption: {excluded_count})")
# --- Reliability metric: how often did we get an unusable response? ---
invalid_percentage = (excluded_count / len(scores_df)) * 100
print(f"Response reliability: {excluded_count}/{len(scores_df)} responses were unusable/corrupted ({invalid_percentage:.1f}%)")
print(f"Average score: {overall_average:.2f} / 2")
print(f"Fully passed (2): {pass_count} ({pass_count/total_tests*100:.0f}%)")
print(f"Partial (1): {partial_count} ({partial_count/total_tests*100:.0f}%)")
print(f"Failed (0): {fail_count} ({fail_count/total_tests*100:.0f}%)")

# --- Breakdown by category ---
print("\n=== BREAKDOWN BY CATEGORY ===")
category_summary = scores_df.groupby("category")["score"].agg(["mean", "count"])
category_summary = category_summary.rename(columns={"mean": "average_score", "count": "num_tests"})
print(category_summary)

category_summary.to_csv("data/scorecard_summary_openrouter.csv")

print("\nDone! Category summary also saved to data/scorecard_summary_openrouter.csv")


## 5. Chart — Gemma by category


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the category summary already generated
summary = pd.read_csv("data/scorecard_summary_gemma.csv")
#draws barcharts
plt.figure(figsize=(8, 5))
bars = plt.bar(summary["category"], summary["average_score"], color="#2a78d6")

plt.ylim(0, 2)
plt.ylabel("Average score (out of 2)")
plt.title("Gemma: Average Score by Category")
plt.xticks(rotation=20, ha="right")

# Add the actual number on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.03, f"{height:.2f}", ha="center")
#safe as png
plt.tight_layout()
plt.savefig("data/chart_gemma_categories.png", dpi=150)
print("Done! Chart saved to data/chart_gemma_categories.png")


## 6. Chart — Nemotron by category


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv("data/scorecard_summary_nemotron.csv")

plt.figure(figsize=(8, 5))
bars = plt.bar(summary["category"], summary["average_score"], color="#2a9d5c")

plt.ylim(0, 2)
plt.ylabel("Average score (out of 2)")
plt.title("Nemotron: Average Score by Category")
plt.xticks(rotation=20, ha="right")

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.03, f"{height:.2f}", ha="center")

plt.tight_layout()
plt.savefig("data/chart_nemotron_categories.png", dpi=150)
print("Done! Chart saved to data/chart_nemotron_categories.png")


## 7. Final comparison — all models side by side

Brings the per-model category scorecards together into one grouped bar chart, so all evaluated
models can be compared category-by-category in a single view. gpt-oss's OpenRouter runs were still
hitting free-tier rate limits at the time this was written (see `02_run_models.ipynb`), so its
scorecard isn't available yet — the cell below only plots whichever model scorecards actually
exist in `data/`, and will pick up gpt-oss automatically once
`data/scorecard_summary_gpt_oss.csv` has been generated the same way as the Gemma/Nemotron ones
above (via `generate_scorecard_openrouter.py`-style aggregation of its scored results).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Each entry is generated the same way as the per-model scorecards above
model_summaries = {
    "Gemma": "data/scorecard_summary_gemma.csv",
    "Nemotron": "data/scorecard_summary_nemotron.csv",
    "gpt-oss": "data/scorecard_summary_gpt_oss.csv",
}

available = {}
for model_name, path in model_summaries.items():
    if os.path.exists(path):
        available[model_name] = pd.read_csv(path).set_index("category")["average_score"]
    else:
        print(f"Skipping {model_name}: {path} not found yet")

comparison = pd.DataFrame(available)
comparison


In [ ]:
categories = comparison.index.tolist()
models = comparison.columns.tolist()
x = np.arange(len(categories))
width = 0.8 / len(models)

plt.figure(figsize=(10, 6))
for i, model in enumerate(models):
    plt.bar(x + i * width, comparison[model], width, label=model)

plt.xticks(x + width * (len(models) - 1) / 2, categories, rotation=20, ha="right")
plt.ylim(0, 2)
plt.ylabel("Average score (out of 2)")
plt.title("Model Comparison by Category")
plt.legend()
plt.tight_layout()
plt.savefig("data/chart_model_comparison.png", dpi=150)
print("Done! Chart saved to data/chart_model_comparison.png")
plt.show()
